In [1]:
import pandas as pd
import sys
from sklearn.metrics import classification_report
from lazypredict.Supervised import LazyClassifier
from imblearn.over_sampling import SMOTE

from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.helpers import Helpers

In [2]:
helper = Helpers()
properties = helper.load_properties()

try:
    RANDOM_STATE = properties['models']['random_state']
except KeyError as ke:
    raise KeyError(f"Missing key in properties file: {str(ke)}") from ke

In [3]:
# Apply global settings
helper.set_global_settings()

In [4]:
dataset_dir = helper.root_dir / "datasets"

train_df = pd.read_csv(dataset_dir / "train_data.csv")
train_df.head()

,person_age,person_education,person_income,person_home_ownership,loan_amount,loan_intent,loan_interest_rate,credit_score,previous_loan_defaults,loan_status
0,-0.64,0,0.33,2,0.08,3,-1.20,-0.86,1,0
1,-0.64,1,-1.13,1,-0.43,1,-1.20,0.50,1,0
2,-0.32,0,-1.21,1,0.08,1,0.11,-0.05,0,0
3,-0.32,1,-0.81,1,-1.11,1,-0.57,-1.20,0,0
4,-1.00,3,0.55,2,0.65,0,0.89,0.01,0,1


In [5]:
val_df = pd.read_csv(dataset_dir / "validation_set.csv")
val_df.head()

,person_age,person_education,person_income,person_home_ownership,loan_amount,loan_intent,loan_interest_rate,credit_score,previous_loan_defaults,loan_status
0,-1.00,3,-0.31,2,1.85,1,-1.20,1.66,0,1
1,-1.42,2,0.10,2,-1.49,0,0.56,1.41,1,0
2,-0.32,2,-1.80,1,-0.43,3,-0.69,0.57,1,0
3,-1.00,2,-0.61,3,-1.72,0,0.06,1.36,0,0
4,0.77,2,-0.16,2,-0.21,5,-0.00,2.24,1,0


In [6]:
target_variable = 'loan_status'

train_df[target_variable].value_counts(normalize=True)

loan_status
0   0.78
1   0.22
Name: proportion, dtype: float64

In [7]:
X_train = train_df.drop(columns=[target_variable])
y_train = train_df[target_variable]

X_val = val_df.drop(columns=[target_variable])
y_val = val_df[target_variable]

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")

Training set size: 30732 samples
Validation set size: 8780 samples


In [8]:
# Calculate distribution before SMOTE
freq_y_train = y_train.value_counts(normalize=True) * 100

# Apply SMOTE to training set
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Calculate distribution after SMOTE
freq_y_train_sm = pd.Series(y_train_sm).value_counts(normalize=True) * 100

# Combine into one dataframe
dist = pd.concat(
    [freq_y_train, freq_y_train_sm], 
    axis=1,
    keys=['Before SMOTE (%)', 'After SMOTE (%)']
).fillna(0).round(2)

# Print markdown table
print(f'Class distribution for {target_variable} (Train set)\n')
print(dist.to_markdown(), '\n')

Class distribution for loan_status (Train set)

|   loan_status |   Before SMOTE (%) |   After SMOTE (%) |
|--------------:|-------------------:|------------------:|
|             0 |              77.71 |                50 |
|             1 |              22.29 |                50 | 



In [9]:
clf = LazyClassifier(
    predictions=True, 
    random_state=RANDOM_STATE
    )

models, predictions = clf.fit(X_train_sm, X_val, y_train_sm, y_val)
models.sort_values(by=['F1 Score', 'Balanced Accuracy', 'ROC AUC', 'Time Taken'], ascending=False)

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 23883, number of negative: 23883
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001124 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 47766, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
XGBClassifier,0.92,0.89,0.89,0.92,0.59
LGBMClassifier,0.91,0.89,0.89,0.91,5.85
RandomForestClassifier,0.91,0.89,0.89,0.91,7.21
ExtraTreesClassifier,0.91,0.89,0.89,0.91,4.55
BaggingClassifier,0.91,0.88,0.88,0.91,2.32
DecisionTreeClassifier,0.89,0.86,0.86,0.89,0.47
SVC,0.87,0.88,0.88,0.88,31.86
KNeighborsClassifier,0.87,0.87,0.87,0.87,1.88
AdaBoostClassifier,0.86,0.88,0.88,0.87,2.70


Top 5 models:

1. `XGBClassifier`

2. `LGBMClassifier`

3. `RandomForestClassifier`

4. `ExtraTreesClassifier`

5. `BaggingClassifier`

In [10]:
models_of_interest = [
    'XGBClassifier', 'LGBMClassifier', 'RandomForestClassifier', 'ExtraTreesClassifier', 'BaggingClassifier'
]

for model in models_of_interest:
    print('\t\t',model,'\n')
    print(classification_report(y_val, predictions[model]),'\n')

		 XGBClassifier 

              precision    recall  f1-score   support

           0       0.96      0.94      0.95      6823
           1       0.79      0.85      0.82      1957

    accuracy                           0.92      8780
   macro avg       0.87      0.89      0.88      8780
weighted avg       0.92      0.92      0.92      8780
 

		 LGBMClassifier 

              precision    recall  f1-score   support

           0       0.95      0.93      0.94      6823
           1       0.78      0.85      0.81      1957

    accuracy                           0.91      8780
   macro avg       0.87      0.89      0.88      8780
weighted avg       0.92      0.91      0.91      8780
 

		 RandomForestClassifier 

              precision    recall  f1-score   support

           0       0.95      0.93      0.94      6823
           1       0.78      0.85      0.81      1957

    accuracy                           0.91      8780
   macro avg       0.87      0.89      0.88      8780
wei

In [11]:
# Combine SMOTE resampled features + target
sampled_df = pd.DataFrame(X_train_sm, columns=X_train.columns)
sampled_df[target_variable] = y_train_sm

# Save to CSV
file_path = dataset_dir / "sampled_train_data.csv"
sampled_df.to_csv(file_path, index=False)